In [ ]:
# %% Deep learning - Section 24.221
#    The LSTM and GRU classes

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% The LSTM class

# Parameters
input_size  =  9  # number of features to extract (e.g., number of data channels)
hidden_size = 16  # number of units in the hidden state
num_layers  =  2  # number of vertical stacks of hidden layers (note: only the final layer gives an output)

# Create LSTM instance (remember that different components in the LSTM have
# different activation functions)
lstm = nn.LSTM(input_size,hidden_size,num_layers)
print(lstm)


In [ ]:
# %% Explore the class

# Data parameters
seq_length = 5
batch_size = 2

# Generate some random data
X = torch.rand(seq_length,batch_size,input_size)

# Initialise hidden states of hidden and cell (typically as zeros)
H = torch.zeros(num_layers,batch_size,hidden_size)
C = torch.zeros(num_layers,batch_size,hidden_size)

# The hidden state input is a tuple (hidden,cell)
hidden_inputs = (H,C)

# Run data through the model and print the output sizes (note how all that comes
# out of the hidden state has the same size)
y,h = lstm(X,hidden_inputs)

print(f' Input shape: {list(X.shape)}')
print(f'Hidden shape: {list(h[0].shape)}')
print(f'  Cell shape: {list(h[1].shape)}')
print(f'Output shape: {list(y.shape)}')
print()

# Check the learnt parameters and their sizes (we have 16x4=64 because the four
# matrices W_i, W_f, W_g, W_o are concatenated)
for p in lstm.named_parameters():
    if 'weight' in p[0]:
        print(f'{p[0]} has size {list(p[1].shape)}')


In [5]:
# %% LSTM class

class LSTMnet(nn.Module):
    def __init__(self,input_size,num_hidden,num_layers):
        super().__init__()

        # Parameters
        self.input_size = input_size
        self.num_hidden = num_hidden
        self.num_layers = num_layers

        # LSTM layers (same notation as RNNs basically) and output layer
        self.lstm = nn.LSTM(input_size,num_hidden,num_layers)
        self.out = nn.Linear(num_hidden,1)

    def forward(self,x):

        print(f'Input: {list(x.shape)}')

        # Pass data through LSTM layer
        y,hidden = self.lstm(x)
        print(f'LSTM-out: {list(y.shape)}')
        print(f'LSTM-hidden: {list(hidden[0].shape)}')
        print(f'LSTM-cell: {list(hidden[1].shape)}')

        # Pass the RNN output through the linear output layer
        o = self.out(y)
        print(f'Output: {list(o.shape)}')

        return o,hidden


In [ ]:
# %% Check the class

# Model instance
net = LSTMnet(input_size,hidden_size,num_layers)
print(net), print()

# Check learnable parameters
for p in net.named_parameters():
    print(f'{p[0]:>20} has size {list(p[1].shape)}')


In [ ]:
# %% Test the class on some random data

X = torch.rand(seq_length,batch_size,input_size)
y = torch.rand(seq_length,batch_size,1)

yHat,h = net(X)

loss_fun = nn.MSELoss()
loss_fun(yHat,y)


In [ ]:
# %% The GRU class

# Create a GRU instance
gru = nn.GRU(input_size,hidden_size,num_layers)
print(gru)


In [ ]:
# %% Explore the class

# Create some random data and a hidden state
X = torch.rand(seq_length,batch_size,input_size)
H = torch.zeros(num_layers,batch_size,hidden_size)

# run some data through the model and show output sizes (no cell states in GRU)
y,h = gru(X,H)

print(f' Input shape: {list(X.shape)}')
print(f'Hidden shape: {list(h.shape)}')
print(f'Output shape: {list(y.shape)}')

# Check the learnt parameters and their sizes (we have 16x3=48 because the three
# matrices W_n, W_r, W_z are also concatenated)
for p in gru.named_parameters():
    print(f'{p[0]:>15} has size {list(p[1].shape)}')
